# CFD: a lid-driven cavity with `%cash_on`

A 2D incompressible Navier-Stokes solver for the classic lid-driven cavity,
followed by post-processing, a grid study and a parameter sweep. It needs
numpy and scipy.

Run it top to bottom once (a few seconds), then follow the **Try this** notes.

**Read the badge, not the clock.** cash prints a badge under every cell it runs.
Open it to see one row per statement: **CACHED** (green) means cash restored the
value instead of running the code, **EXECUTED** (ochre) means the code ran.
Printed output is no evidence either way: a restored statement replays what it
printed.

In [ ]:
import cash
%cash_on

In [ ]:
from pathlib import Path

import numpy as np
from scipy import sparse
from scipy.sparse.linalg import factorized, spsolve

## 1. Domain and physical parameters

The lid (top wall) moves at velocity $U = 1$; the other walls are stationary.
The flow obeys the 2D incompressible Navier-Stokes equations:

$$\frac{\partial \mathbf{u}}{\partial t} + (\mathbf{u} \cdot \nabla)\mathbf{u} = -\frac{1}{\rho}\nabla p + \nu \nabla^2 \mathbf{u}, \qquad \nabla \cdot \mathbf{u} = 0$$

The Reynolds number $Re = UL/\nu$ sets the flow regime.

In [ ]:
# Physical parameters — change Re to explore different flow regimes
Re = 100          # Reynolds number (try 100, 400, 1000)
U_lid = 1.0       # Lid velocity [m/s]
L = 1.0           # Cavity length [m]
nu = U_lid * L / Re  # Kinematic viscosity
rho = 1.0         # Density [kg/m³]
print(f"Reynolds number: {Re}")
print(f"Kinematic viscosity: {nu:.6f} m²/s")

# Computational grid
N = 41            # Grid points in each direction (41x41)
dx = L / (N - 1)
dy = L / (N - 1)
dt = 0.001        # Time step [s]
n_steps = 10_000   # Number of time steps (10 s of flow)
print(f"Grid: {N}x{N}, dx=dy={dx:.4f}, dt={dt}, steps={n_steps}")

# Create coordinate arrays
x = np.linspace(0, L, N)
y = np.linspace(0, L, N)
X, Y = np.meshgrid(x, y)
print(f"Domain: [{x[0]}, {x[-1]}] x [{y[0]}, {y[-1]}]")

## 2. Solver functions

`@cash.pure` declares that a function has no side effects, so cash trusts it
instead of analysing its body. Each function here returns new arrays and
changes nothing it is given.

The pressure Poisson equation:
$$\nabla^2 p = \frac{\rho}{\Delta t} \left( \frac{\partial u}{\partial x} + \frac{\partial v}{\partial y} \right)$$

In [ ]:
@cash.pure
def build_laplacian_2d(n, dx, dy):
    """Build the 2D Laplacian operator as a sparse matrix.

    Uses 5-point stencil: ∇²f ≈ (f_{i+1,j} + f_{i-1,j} + f_{i,j+1} + f_{i,j-1} - 4f_{i,j}) / h²
    """
    n2 = n * n
    diags = np.zeros((5, n2))

    # Main diagonal
    diags[2, :] = -2.0 / dx**2 - 2.0 / dy**2
    # Off-diagonals (x-direction)
    diags[1, :] = 1.0 / dx**2  # i+1
    diags[3, :] = 1.0 / dx**2  # i-1
    # Off-diagonals (y-direction)
    diags[0, :] = 1.0 / dy**2  # j+1
    diags[4, :] = 1.0 / dy**2  # j-1

    offsets = [n, 1, 0, -1, -n]
    A = sparse.diags(diags, offsets, shape=(n2, n2), format='csc')
    return A

@cash.pure
def compute_divergence(u, v, dx, dy):
    """Compute velocity divergence field: ∂u/∂x + ∂v/∂y."""
    div = np.zeros_like(u)
    div[1:-1, 1:-1] = (
        (u[1:-1, 2:] - u[1:-1, :-2]) / (2 * dx) +
        (v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * dy)
    )
    return div

@cash.pure
def apply_boundary_conditions(u, v, U_lid):
    """Apply no-slip walls + moving lid (top boundary). Returns new arrays."""
    u, v = u.copy(), v.copy()
    # No-slip on all walls
    u[0, :] = 0;  u[-1, :] = 0;  u[:, 0] = 0;  u[:, -1] = 0
    v[0, :] = 0;  v[-1, :] = 0;  v[:, 0] = 0;  v[:, -1] = 0
    # Moving lid (top wall)
    u[-1, :] = U_lid
    return u, v

print("Solver functions defined")

## 3. Sparse operators

The Laplacian and its LU factorisation depend only on the grid (`N`, `dx`,
`dy`), so they stay cached while you change anything else.

In [ ]:
A_laplacian = build_laplacian_2d(N, dx, dy)
print(f"Laplacian matrix: {A_laplacian.shape}, {A_laplacian.nnz} non-zeros")

# Pre-factor the matrix for fast repeated solves
from scipy.sparse.linalg import factorized
pressure_solve = factorized(A_laplacian)
print("LU factorization done")

## 4. Time stepping

Chorin's projection method: an explicit advection-diffusion step, a pressure
Poisson solve, then a velocity correction.

The solver lives in a function that returns its results, so the call is a
single statement that cash can cache and restore. Written as a bare loop that
appends to a list created before it, it would run every time: cash does not
cache a statement whose effect is to change an existing object in place.

In [ ]:
def run_cavity(N, L, dt, nu, rho, U_lid, n_steps):
    """March the cavity flow forward n_steps. Returns u, v, p and the residual history."""
    dx = dy = L / (N - 1)
    solve = factorized(build_laplacian_2d(N, dx, dy))
    u = np.zeros((N, N))  # x-velocity
    v = np.zeros((N, N))  # y-velocity
    p = np.zeros((N, N))  # pressure
    u, v = apply_boundary_conditions(u, v, U_lid)
    residual_history = []

    for step in range(n_steps):
        # 1. Compute advection terms (nonlinear convection)
        u_adv = u.copy()
        v_adv = v.copy()

        u_adv[1:-1, 1:-1] = (u[1:-1, 1:-1]
            - dt * u[1:-1, 1:-1] * (u[1:-1, 2:] - u[1:-1, :-2]) / (2 * dx)
            - dt * v[1:-1, 1:-1] * (u[2:, 1:-1] - u[:-2, 1:-1]) / (2 * dy)
            + dt * nu * (
                (u[1:-1, 2:] - 2*u[1:-1, 1:-1] + u[1:-1, :-2]) / dx**2 +
                (u[2:, 1:-1] - 2*u[1:-1, 1:-1] + u[:-2, 1:-1]) / dy**2
            ))

        v_adv[1:-1, 1:-1] = (v[1:-1, 1:-1]
            - dt * u[1:-1, 1:-1] * (v[1:-1, 2:] - v[1:-1, :-2]) / (2 * dx)
            - dt * v[1:-1, 1:-1] * (v[2:, 1:-1] - v[:-2, 1:-1]) / (2 * dy)
            + dt * nu * (
                (v[1:-1, 2:] - 2*v[1:-1, 1:-1] + v[1:-1, :-2]) / dx**2 +
                (v[2:, 1:-1] - 2*v[1:-1, 1:-1] + v[:-2, 1:-1]) / dy**2
            ))

        # 2. Pressure Poisson solve
        div = compute_divergence(u_adv, v_adv, dx, dy)
        rhs = (rho / dt) * div.flatten()
        p_flat = solve(rhs)
        p = p_flat.reshape((N, N))

        # 3. Velocity correction (projection)
        u[1:-1, 1:-1] = u_adv[1:-1, 1:-1] - (dt / rho) * (p[1:-1, 2:] - p[1:-1, :-2]) / (2 * dx)
        v[1:-1, 1:-1] = v_adv[1:-1, 1:-1] - (dt / rho) * (p[2:, 1:-1] - p[:-2, 1:-1]) / (2 * dy)

        # Apply boundary conditions
        u, v = apply_boundary_conditions(u, v, U_lid)

        # Track convergence
        residual = np.sqrt(np.mean(div[1:-1, 1:-1]**2))
        residual_history.append(residual)

        if step % 1000 == 0:
            print(f"  Step {step:4d}/{n_steps}: residual={residual:.2e}, "
                  f"|u|_max={np.max(np.abs(u)):.4f}, |v|_max={np.max(np.abs(v)):.4f}")

    return u, v, p, residual_history


u, v, p, residual_history = run_cavity(N, L, dt, nu, rho, U_lid, n_steps)
print(f"Final residual: {residual_history[-1]:.2e}")

## 5. Derived quantities

Vorticity, stream function and kinetic energy from the final velocity field.

In [ ]:
# Vorticity field  ω = ∂v/∂x - ∂u/∂y
vorticity = np.zeros((N, N))
vorticity[1:-1, 1:-1] = (
    (v[1:-1, 2:] - v[1:-1, :-2]) / (2 * dx) -
    (u[2:, 1:-1] - u[:-2, 1:-1]) / (2 * dy)
)
print(f"Vorticity: min={vorticity.min():.4f}, max={vorticity.max():.4f}")

# Stream function via Poisson solve  ∇²ψ = -ω
psi_flat = spsolve(A_laplacian, -vorticity.flatten())
stream_function = psi_flat.reshape((N, N))
print(f"Stream function: min={stream_function.min():.6f}, max={stream_function.max():.6f}")

# Kinetic energy field  E = 0.5 * ρ * (u² + v²)
kinetic_energy = 0.5 * rho * (u**2 + v**2)
total_KE = np.sum(kinetic_energy) * dx * dy
print(f"Total kinetic energy: {total_KE:.6f} J")

## 6. Branching on the flow regime

Only the branch that runs is cached.

**Try this:** change `Re` in section 1 to `1000` and run the cells down to
here. This cell now takes the transitional branch, and the badge shows only
that branch's statements.

In [ ]:
print(f"\n{'='*50}")
print(f"Flow Regime Analysis for Re = {Re}")
print(f"{'='*50}")

if Re < 200:
    # Laminar regime — smooth streamlines, predictable vortex
    regime = "laminar"

    # Find primary vortex center (location of min stream function)
    iy, ix = np.unravel_index(np.argmin(stream_function), stream_function.shape)
    vortex_x, vortex_y = x[ix], y[iy]
    vortex_strength = vorticity[iy, ix]

    # Compute velocity profile along vertical centerline
    mid_x = N // 2
    u_centerline = u[:, mid_x]

    print(f"Regime: {regime}")
    print(f"Primary vortex center: ({vortex_x:.3f}, {vortex_y:.3f})")
    print(f"Vortex strength: {vortex_strength:.4f}")
    print(f"Centerline u range: [{u_centerline.min():.4f}, {u_centerline.max():.4f}]")

elif Re < 500:
    # Moderate Re — check for secondary corner vortices
    regime = "moderate"

    # Bottom-left corner vortex detection
    corner_region = stream_function[:N//4, :N//4]
    has_secondary = np.any(corner_region > 0) and np.any(corner_region < 0)

    # Enstrophy (integral of vorticity²) — measures flow complexity
    enstrophy = 0.5 * np.sum(vorticity**2) * dx * dy

    print(f"Regime: {regime}")
    print(f"Secondary corner vortex detected: {has_secondary}")
    print(f"Enstrophy: {enstrophy:.6f}")

else:
    # High Re — transitional, compute energy spectrum
    regime = "transitional"

    # 2D FFT of velocity magnitude for spectral analysis
    speed = np.sqrt(u**2 + v**2)
    speed_fft = np.fft.fft2(speed[1:-1, 1:-1])
    power_spectrum = np.abs(speed_fft)**2

    # Radially-averaged energy spectrum
    ny, nx_fft = power_spectrum.shape
    kx = np.fft.fftfreq(nx_fft, d=dx)
    ky = np.fft.fftfreq(ny, d=dy)
    KX, KY = np.meshgrid(kx, ky)
    K_mag = np.sqrt(KX**2 + KY**2)

    # Bin the spectrum
    k_bins = np.linspace(0, K_mag.max(), 20)
    spectrum_binned = np.zeros(len(k_bins) - 1)
    for i in range(len(k_bins) - 1):
        mask = (K_mag >= k_bins[i]) & (K_mag < k_bins[i+1])
        if mask.any():
            spectrum_binned[i] = np.mean(power_spectrum[mask])

    print(f"Regime: {regime}")
    print(f"Peak wavenumber: {k_bins[np.argmax(spectrum_binned)+1]:.1f}")
    print(f"Total spectral energy: {np.sum(spectrum_binned):.2e}")

print(f"\nKinetic energy: {total_KE:.6f} J")
print(f"Max velocity magnitude: {np.max(np.sqrt(u**2 + v**2)):.4f} m/s")

## 7. Grid convergence study

Each resolution is one iteration of the loop, and cash caches each iteration on
its own.

**Try this:** add `51` to `resolutions` and run the cell again. The four
resolutions you already had are restored; only the new one runs.

In [ ]:
resolutions = [11, 21, 31, 41]
convergence_results = {}

for n_grid in resolutions:
    u_c, v_c, _, _ = run_cavity(n_grid, L, dt, nu, rho, U_lid, 1000)
    mid = n_grid // 2
    convergence_results[n_grid] = {"u_center": u_c[mid, mid], "max_speed": np.max(np.sqrt(u_c**2 + v_c**2))}
    print(f"  N={n_grid:3d}: u_center={u_c[mid, mid]:.6f}, max_speed={convergence_results[n_grid]['max_speed']:.4f}")

# Richardson extrapolation between two finest grids
if len(resolutions) >= 2:
    u_fine = convergence_results[resolutions[-1]]['u_center']
    u_coarse = convergence_results[resolutions[-2]]['u_center']
    r = resolutions[-1] / resolutions[-2]  # refinement ratio
    order = np.log(abs((convergence_results[resolutions[-3]]['u_center'] - u_coarse) / 
                       (u_coarse - u_fine + 1e-15))) / np.log(r) if len(resolutions) >= 3 else 2.0
    u_exact_est = u_fine + (u_fine - u_coarse) / (r**order - 1)
    print(f"\nRichardson extrapolation: u_exact ≈ {u_exact_est:.6f} (order ≈ {order:.1f})")

## 8. Perturbed initial conditions

The perturbation comes from a seeded generator, so a re-run produces the same
field and the cached value is the one a fresh run would give.

In [ ]:
# Perturbed initial condition for studying sensitivity
rng = np.random.default_rng(seed=42)
perturbation_amplitude = 0.01

u_perturbed = np.zeros((N, N)) + perturbation_amplitude * rng.standard_normal((N, N))
v_perturbed = np.zeros((N, N)) + perturbation_amplitude * rng.standard_normal((N, N))
u_perturbed, v_perturbed = apply_boundary_conditions(u_perturbed, v_perturbed, U_lid)

# Run 200 steps with perturbed IC
for s in range(200):
    u_tmp = u_perturbed.copy()
    v_tmp = v_perturbed.copy()
    u_tmp[1:-1,1:-1] = (u_perturbed[1:-1,1:-1]
        - dt*u_perturbed[1:-1,1:-1]*(u_perturbed[1:-1,2:]-u_perturbed[1:-1,:-2])/(2*dx)
        - dt*v_perturbed[1:-1,1:-1]*(u_perturbed[2:,1:-1]-u_perturbed[:-2,1:-1])/(2*dy)
        + dt*nu*((u_perturbed[1:-1,2:]-2*u_perturbed[1:-1,1:-1]+u_perturbed[1:-1,:-2])/dx**2
                +(u_perturbed[2:,1:-1]-2*u_perturbed[1:-1,1:-1]+u_perturbed[:-2,1:-1])/dy**2))
    v_tmp[1:-1,1:-1] = (v_perturbed[1:-1,1:-1]
        - dt*u_perturbed[1:-1,1:-1]*(v_perturbed[1:-1,2:]-v_perturbed[1:-1,:-2])/(2*dx)
        - dt*v_perturbed[1:-1,1:-1]*(v_perturbed[2:,1:-1]-v_perturbed[:-2,1:-1])/(2*dy)
        + dt*nu*((v_perturbed[1:-1,2:]-2*v_perturbed[1:-1,1:-1]+v_perturbed[1:-1,:-2])/dx**2
                +(v_perturbed[2:,1:-1]-2*v_perturbed[1:-1,1:-1]+v_perturbed[:-2,1:-1])/dy**2))

    div_p = compute_divergence(u_tmp, v_tmp, dx, dy)
    p_p = pressure_solve((rho/dt)*div_p.flatten()).reshape((N, N))
    u_perturbed[1:-1,1:-1] = u_tmp[1:-1,1:-1] - (dt/rho)*(p_p[1:-1,2:]-p_p[1:-1,:-2])/(2*dx)
    v_perturbed[1:-1,1:-1] = v_tmp[1:-1,1:-1] - (dt/rho)*(p_p[2:,1:-1]-p_p[:-2,1:-1])/(2*dy)
    u_perturbed, v_perturbed = apply_boundary_conditions(u_perturbed, v_perturbed, U_lid)


# Compare with unperturbed solution
u_diff = np.max(np.abs(u_perturbed - u))
v_diff = np.max(np.abs(v_perturbed - v))
print(f"Max velocity difference from unperturbed:")
print(f"  |Δu|_max = {u_diff:.6e}")
print(f"  |Δv|_max = {v_diff:.6e}")
print(f"  Perturbation amplification factor: {max(u_diff, v_diff) / perturbation_amplitude:.2f}x")

## 9. Saving and reloading results

Writing a file is a side effect, so cash never caches the saving cell. Reading
the file with `np.load` is tracked: if the file changes on disk, the reading
cell runs again.

In [ ]:
# Save simulation results to disk
results_dir = Path(".cash_cfd_results")
results_dir.mkdir(exist_ok=True)

result_file = results_dir / f"cavity_Re{Re}_N{N}.npz"
np.savez(result_file,
         u=u, v=v, p=p, vorticity=vorticity,
         stream_function=stream_function,
         x=x, y=y, Re=Re, N=N)
file_size = result_file.stat().st_size
print(f"Saved results to: {result_file}")
print(f"File size: {file_size / 1024:.1f} KB")

In [ ]:
loaded = np.load(result_file)
u_loaded = loaded['u']
v_loaded = loaded['v']

# Verify integrity
u_match = np.allclose(u, u_loaded)
v_match = np.allclose(v, v_loaded)
print(f"Loaded Re={int(loaded['Re'])}, N={int(loaded['N'])}")
print(f"Integrity check: u={u_match}, v={v_match}")

# Compute derived quantity from loaded data
speed_loaded = np.sqrt(u_loaded**2 + v_loaded**2)
print(f"Max speed from loaded data: {speed_loaded.max():.4f} m/s")

## 10. Convergence history

In [ ]:
# Convergence diagnostics
residuals = np.array(residual_history)
print(f"Convergence History (Re={Re}, N={N}, {n_steps} steps):")
print(f"  Initial residual:  {residuals[0]:.6e}")
print(f"  Final residual:    {residuals[-1]:.6e}")
print(f"  Reduction factor:  {residuals[0] / (residuals[-1] + 1e-15):.1f}x")
print(f"  Min residual:      {residuals.min():.6e} (at step {residuals.argmin()})")

# Compute convergence rate (exponential fit to last 50% of history)
half = len(residuals) // 2
if residuals[half:].min() > 0:
    log_res = np.log(residuals[half:])
    steps_arr = np.arange(half, len(residuals))
    coeffs = np.polyfit(steps_arr, log_res, 1)
    convergence_rate = coeffs[0]
    print(f"  Convergence rate:  {convergence_rate:.6f} (exponential decay constant)")
    print(f"  Half-life:         {-np.log(2)/convergence_rate:.0f} steps")
else:
    print(f"  Residual reached machine zero — fully converged!")

# Velocity field statistics
speed = np.sqrt(u**2 + v**2)

print(f"\nVelocity Field Statistics:")
print(f"  Mean speed:     {speed.mean():.6f} m/s")
print(f"  Max speed:      {speed.max():.6f} m/s")
print(f"  RMS velocity:   {np.sqrt(np.mean(u**2 + v**2)):.6f} m/s")
print(f"  Max |u|:        {np.max(np.abs(u)):.6f} m/s")
print(f"  Max |v|:        {np.max(np.abs(v)):.6f} m/s")

## What cash did this session

Restart the kernel and run everything again: the slow cached results come back
from disk, and `%cash_stats` shows how much time that saved.

In [ ]:
%cash_stats